In [ ]:
import os, sys

print("Aantal paden in sys.path:", len(sys.path))
# Bepaal de parent-werkmap
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

#Loop door alle submappen en voeg ze toe aan sys.path
for dirpath, dirnames, filenames in os.walk(parent_dir):
    if dirpath not in sys.path:
        sys.path.insert(0, dirpath)

print("Aantal paden in sys.path:", len(sys.path))

In [ ]:
import sys
import os
from pathlib import Path

# Add the Tygron folder to the Python path to access Libraries
# In a Jupyter Notebook, __file__ is not defined. Use os.getcwd() instead.
# Assumes the notebook is in a subdirectory of the project root (Tygron folder).
tygron_path = Path(os.getcwd()).parent
sys.path.insert(0, str(tygron_path))

from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import drawImageAndFeatureMasks
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
from Libraries.inference_training import createModelInstance, testInference
from Libraries.engine import evaluate, train_one_epoch
import Libraries.utils as utils
import torch
import random
import math

# Now import paths from the Tygron folder
from paths import EVAL, MODELS_DIR, TEST, TRAIN, PROJECT_ROOT

In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [ ]:
def vector_angle(r1, r2):
    min = len(r1) if len(r2) > len(r1) else len(r2)
    i = 0 
    g1 = []
    g2 = []
    while i < min - 1:
        g1.append((r1[i + 1] - r1[i]))
        g2.append((r2[i + 1] - r1[i]))
        i += 1

    min = len(g1) if len(g2) > len(g1) else len(g2)
    i = 0
    dot = 0
    mag1 = 0
    mag2 = 0
    while i < min:
        dot += g1[i]*g2[i]
        mag1 += g1[i]*g1[i]
        mag2 += g2[i]*g2[i]

        i += 1
    mag1 = math.sqrt(mag1)
    mag2 = math.sqrt(mag2)

    return math.acos(dot/(mag1*mag2))  # in radians

In [ ]:
def load_model(path:str, config=None):
    """
    Load a trained PyTorch instance segmentation model from disk.

    This function initializes a model using the provided or default configuration and loads
    the saved model weights from the specified path.

    Parameters:
    -----------
    path : str
        Path to the saved PyTorch model.
    config : Configuration, optional
        Predefined Configuration object. If not provided, a default configuration will be created.

    Returns:
    --------
    model : torch.nn.Module
        The loaded PyTorch model ready for inference or further training.
    config : Configuration
        The Configuration object used to initialize the model.

    Notes:
    ------
    - If `config` is None, a new Configuration object will be initialized.
    - The function assumes the model architecture is defined in accordance with the Configuration.
    - Ensure the configuration matches the model's training parameters to avoid shape mismatches.
    """
    if not config:
        print("No config found (func load_model)")
        config = Configuration()



    # Check if the path is onnx ifso handle differently
    strpath = str(path).lower()
    if strpath.endswith(".onnx"):
        print("Loading ONNX model")
        model = loadONNX(config)
    else:
        model = createModelInstance(config)
        if path is not None:
            model.load_state_dict(torch.load(path, weights_only=False, map_location=torch.device('cpu')))

    return model, config

In [ ]:
def load_default_config():
    """
    Create and return a default Configuration object with preset parameters.

    This function initializes a Configuration instance with standard settings suitable for
    instance segmentation training and inference, including input sizes, model info, ONNX metadata,
    and legend entries.

    Parameters:
    -----------
    None

    Returns:
    --------
    config : Configuration
        A Configuration object initialized with default values for model training and export.

    Notes:
    ------
    - The default input image size is set to 250x250 pixels.
    - The model is configured for 3 input channels and 3 classes (including background).
    - ONNX metadata thresholds are set to typical defaults for score, mask, and stride.
    - The legend includes a "Background" entry with a transparent color.
    """
    config = Configuration()
    config.setIsCrowd(False)
    config.setModelName("C:\\Users\\Tomkr\\Jaar2BlokD\\Tygron\\Tygron\\Models\\combo_sets_model.pt")
    config.setFilePrefix("")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.addLegendEntry("Background", 0, "#00000000")
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    
    return config

In [ ]:
model_path = str(MODELS_DIR) + "/" + "final_epoch_5.pt"
trainPath = str(PROJECT_ROOT) + "/onderzoek/Ref_set"
config = load_default_config()
model, config = load_model(model_path,config)
config.setDatasetPaths(trainPath= trainPath, testPath=trainPath)  # function does not reach test path, therefore this does not cause issues

refSet = ImageDataset(config, True, createTransforms(False))
placeholder = ImageDataset(config, False, createTransforms(False))

ref_values = trainModel(config, refSet, placeholder, model=model, test= True)

directory = str(PROJECT_ROOT) + "/onderzoek/Test_sets"
result_dict = {}
for name in os.listdir(directory):
    test_directory = directory + "/" + name
    print(test_directory)
    config.setDatasetPaths(trainPath=test_directory, testPath=test_directory)
    testSet = ImageDataset(config, True, createTransforms(False))
    placeholder = ImageDataset(config, False, createTransforms(False))

    test_values = trainModel(config, testSet, placeholder, model=model, test= True)
    result_dict[name] = vector_angle(ref_values, test_values)

for name in result_dict.keys():
    print(name, " gradient angle: ", result_dict[name], "\n")

    


    